In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [2]:
RAW_FILE = "online_retail_II.xlsx"

df_2009 = pd.read_excel(RAW_FILE, sheet_name="Year 2009-2010")
df_2010 = pd.read_excel(RAW_FILE, sheet_name="Year 2010-2011")

df = pd.concat([df_2009, df_2010], ignore_index=True)

print(f"Combined shape: {df.shape}")
df.head()

Combined shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [4]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print(f"Duplicate rows: {df.duplicated().sum()}")
print()
print(f"Unique invoices: {df['Invoice'].nunique()}")
print(f"Unique customers: {df['Customer ID'].nunique()}")
print(f"Unique products: {df['StockCode'].nunique()}")
print(f"Unique countries: {df['Country'].nunique()}")

Missing values per column:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Duplicate rows: 34335

Unique invoices: 53628
Unique customers: 5942
Unique products: 5305
Unique countries: 43


In [5]:
# Check for cancelled orders (Invoice numbers starting with 'C')
cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
print(f"Cancelled order line items: {len(cancelled)}")

# Check for negative/zero quantities and prices
print(f"Rows with Quantity <= 0: {(df['Quantity'] <= 0).sum()}")
print(f"Rows with Price <= 0: {(df['Price'] <= 0).sum()}")

Cancelled order line items: 19494
Rows with Quantity <= 0: 22950
Rows with Price <= 0: 6207


In [6]:
df_clean = df.copy()

# 1. Drop missing Customer ID
before = len(df_clean)
df_clean = df_clean.dropna(subset=['Customer ID'])
print(f"Dropped {before - len(df_clean)} rows with missing Customer ID")

# 2. Remove cancelled orders
before = len(df_clean)
df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]
print(f"Dropped {before - len(df_clean)} cancelled order rows")

# 3. Remove non-positive quantity/price
before = len(df_clean)
# df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]
df_clean['Price'] = df_clean['Price'].round(2)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]
print(f"Dropped {before - len(df_clean)} rows with non-positive quantity/price")

# 4. Remove duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Dropped {before - len(df_clean)} duplicate rows")

# 5. Fix types
df_clean['Invoice'] = df_clean['Invoice'].astype(str).str.strip()
df_clean['StockCode'] = df_clean['StockCode'].astype(str).str.strip()
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)
df_clean['Description'] = df_clean['Description'].astype(str).str.strip()
df_clean['Country'] = df_clean['Country'].astype(str).str.strip()

print(f"\nFinal cleaned shape: {df_clean.shape}")
df_clean.head()

Dropped 243007 rows with missing Customer ID
Dropped 18744 cancelled order rows
Dropped 89 rows with non-positive quantity/price
Dropped 26124 duplicate rows

Final cleaned shape: (779407, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [7]:
# Add column: LineTotal = Quantity * Price
df_clean['LineTotal'] = df_clean['Quantity'] * df_clean['Price']

# Sanity check on cleaned data
print(df_clean[['Quantity', 'Price', 'LineTotal']].describe())

            Quantity          Price      LineTotal
count  779407.000000  779407.000000  779407.000000
mean       13.489658       3.218562      22.292338
std       145.857486      29.676478     227.429676
min         1.000000       0.030000       0.060000
25%         2.000000       1.250000       4.950000
50%         6.000000       1.950000      12.480000
75%        12.000000       3.750000      19.800000
max     80995.000000   10953.500000  168469.600000


In [8]:
# Quick validation: no nulls left in key columns, no negative values
assert df_clean['Customer ID'].isnull().sum() == 0
assert (df_clean['Quantity'] > 0).all()
assert (df_clean['Price'] > 0).all()
print("Validation passed: no missing Customer IDs, all quantities/prices positive.")

Validation passed: no missing Customer IDs, all quantities/prices positive.


In [9]:
# dim_customer
dim_customer = df_clean[['Customer ID', 'Country']].drop_duplicates(subset=['Customer ID']).reset_index(drop=True)
dim_customer.columns = ['customer_id', 'country']
print(f"dim_customer: {len(dim_customer)} rows")
dim_customer.head()

dim_customer: 5878 rows


,customer_id,country
0,13085,United Kingdom
1,13078,United Kingdom
2,15362,United Kingdom
3,18102,United Kingdom
4,12682,France


In [10]:
# dim_product
dim_product = df_clean[['StockCode', 'Description']].drop_duplicates(subset=['StockCode']).reset_index(drop=True)
dim_product.columns = ['stock_code', 'description']
print(f"dim_product: {len(dim_product)} rows")
dim_product.head()

dim_product: 4630 rows


,stock_code,description
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,79323P,PINK CHERRY LIGHTS
2,79323W,WHITE CHERRY LIGHTS
3,22041,"RECORD FRAME 7"" SINGLE SIZE"
4,21232,STRAWBERRY CERAMIC TRINKET BOX


In [11]:
# dim_date - build from the min/max date range in the data
date_range = pd.date_range(start=df_clean['InvoiceDate'].dt.date.min(),
end=df_clean['InvoiceDate'].dt.date.max(), freq='D')

dim_date = pd.DataFrame({'full_date': date_range})
dim_date['date_key'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['month_name'] = dim_date['full_date'].dt.strftime('%B')
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['day_of_week'] = dim_date['full_date'].dt.strftime('%A')

dim_date = dim_date[['date_key', 'full_date', 'year', 'month', 'month_name', 'quarter', 'day_of_week']]
print(f"dim_date: {len(dim_date)} rows")
dim_date.head()


dim_date: 739 rows


,date_key,full_date,year,month,month_name,quarter,day_of_week
0,20091201,2009-12-01,2009,12,December,4,Tuesday
1,20091202,2009-12-02,2009,12,December,4,Wednesday
2,20091203,2009-12-03,2009,12,December,4,Thursday
3,20091204,2009-12-04,2009,12,December,4,Friday
4,20091205,2009-12-05,2009,12,December,4,Saturday


In [12]:
fact_sales = df_clean[['Invoice', 'InvoiceDate', 'Customer ID', 'StockCode', 'Quantity', 'Price', 'LineTotal']].copy()
fact_sales['date_key'] = fact_sales['InvoiceDate'].dt.strftime('%Y%m%d').astype(int)
fact_sales.columns = ['invoice', 'invoice_date', 'customer_id', 'stock_code', 'quantity', 'price', 'line_total', 'date_key']

print(f"fact_sales: {len(fact_sales)} rows")
fact_sales.head()


fact_sales: 779407 rows


,invoice,invoice_date,customer_id,stock_code,quantity,price,line_total,date_key
0,489434,2009-12-01 07:45:00,13085,85048,12,6.95,83.4,20091201
1,489434,2009-12-01 07:45:00,13085,79323P,12,6.75,81.0,20091201
2,489434,2009-12-01 07:45:00,13085,79323W,12,6.75,81.0,20091201
3,489434,2009-12-01 07:45:00,13085,22041,48,2.10,100.8,20091201
4,489434,2009-12-01 07:45:00,13085,21232,24,1.25,30.0,20091201


In [13]:
import os

import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

# Credentials live in .env next to this notebook, not in the notebook itself
load_dotenv()

required = ['DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD']
missing = [k for k in required if not os.getenv(k)]
assert not missing, f"Missing from .env: {', '.join(missing)}"

conn = psycopg2.connect(
    host=os.getenv('DB_HOST'),
    port=int(os.getenv('DB_PORT')),
    dbname=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD')
)
cur = conn.cursor()
print(f"Connected to PostgreSQL: {os.getenv('DB_NAME')} @ {os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}")


Connected to PostgreSQL: retail_dw @ localhost:5432


In [14]:
# Load dim_date
rows = list(dim_date.itertuples(index=False, name=None))
execute_values(cur, """
    INSERT INTO dim_date (date_key, full_date, year, month, month_name, quarter, day_of_week)
    VALUES %s
    ON CONFLICT (date_key) DO NOTHING
""", rows)
conn.commit()
print(f"Loaded {len(rows)} rows into dim_date")


Loaded 739 rows into dim_date


In [15]:
# Load dim_customer
rows = list(dim_customer.itertuples(index=False, name=None))
execute_values(cur, """
    INSERT INTO dim_customer (customer_id, country)
    VALUES %s
    ON CONFLICT (customer_id) DO NOTHING
""", rows)
conn.commit()
print(f"Loaded {len(rows)} rows into dim_customer")

Loaded 5878 rows into dim_customer


In [16]:
# Load dim_product
rows = list(dim_product.itertuples(index=False, name=None))
execute_values(cur, """
    INSERT INTO dim_product (stock_code, description)
    VALUES %s
    ON CONFLICT (stock_code) DO NOTHING
""", rows)
conn.commit()
print(f"Loaded {len(rows)} rows into dim_product")


Loaded 4630 rows into dim_product


In [17]:
# Pull surrogate key lookups from Postgres
customer_lookup = pd.read_sql("SELECT customer_key, customer_id FROM dim_customer", conn)
product_lookup = pd.read_sql("SELECT product_key, stock_code FROM dim_product", conn)

# Join onto fact_sales to replace natural keys with surrogate keys
fact_load = fact_sales.merge(customer_lookup, on='customer_id', how='left')
fact_load = fact_load.merge(product_lookup, on='stock_code', how='left')

# Drop rows that failed to match (shouldn't happen if dimensions loaded correctly)
before = len(fact_load)
fact_load = fact_load.dropna(subset=['customer_key', 'product_key'])
print(f"Dropped {before - len(fact_load)} unmatched rows")

fact_load = fact_load[['invoice', 'date_key', 'customer_key', 'product_key', 'quantity', 'price', 'line_total']]
fact_load['customer_key'] = fact_load['customer_key'].astype(int)
fact_load['product_key'] = fact_load['product_key'].astype(int)
fact_load.head()


Dropped 0 unmatched rows


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16884\1961492944.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customer_lookup = pd.read_sql("SELECT customer_key, customer_id FROM dim_customer", conn)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16884\1961492944.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  product_lookup = pd.read_sql("SELECT product_key, stock_code FROM dim_product", conn)


,invoice,date_key,customer_key,product_key,quantity,price,line_total
0,489434,20091201,1,1,12,6.95,83.4
1,489434,20091201,1,2,12,6.75,81.0
2,489434,20091201,1,3,12,6.75,81.0
3,489434,20091201,1,4,48,2.10,100.8
4,489434,20091201,1,5,24,1.25,30.0


In [ ]:
# Load fact_sales in batches (large table — batch inserts for performance)
rows = list(fact_load.itertuples(index=False, name=None))

batch_size = 10000
for i in range(0, len(rows), batch_size):
    batch = rows[i:i+batch_size]
    execute_values(cur, """
        INSERT INTO fact_sales (invoice, date_key, customer_key, product_key, quantity, price, line_total)
        VALUES %s
    """, batch)
    conn.commit()
    print(f"Loaded rows {i} to {i+len(batch)}")

print(f"\nTotal fact_sales rows loaded: {len(rows)}")

Loaded rows 0 to 10000
Loaded rows 10000 to 20000
Loaded rows 20000 to 30000
Loaded rows 30000 to 40000
Loaded rows 40000 to 50000
Loaded rows 50000 to 60000
Loaded rows 60000 to 70000
Loaded rows 70000 to 80000
Loaded rows 80000 to 90000
Loaded rows 90000 to 100000
Loaded rows 100000 to 110000
Loaded rows 110000 to 120000
Loaded rows 120000 to 130000
Loaded rows 130000 to 140000
Loaded rows 140000 to 150000
Loaded rows 150000 to 160000
Loaded rows 160000 to 170000
Loaded rows 170000 to 180000
Loaded rows 180000 to 190000
Loaded rows 190000 to 200000
Loaded rows 200000 to 210000
Loaded rows 210000 to 220000
Loaded rows 220000 to 230000
Loaded rows 230000 to 240000
Loaded rows 240000 to 250000
Loaded rows 250000 to 260000
Loaded rows 260000 to 270000
Loaded rows 270000 to 280000
Loaded rows 280000 to 290000
Loaded rows 290000 to 300000
Loaded rows 300000 to 310000
Loaded rows 310000 to 320000
Loaded rows 320000 to 330000
Loaded rows 330000 to 340000
Loaded rows 340000 to 350000
Loaded r

In [19]:
verify = pd.read_sql("""
    SELECT
        (SELECT COUNT(*) FROM dim_date) AS dim_date_count,
        (SELECT COUNT(*) FROM dim_customer) AS dim_customer_count,
        (SELECT COUNT(*) FROM dim_product) AS dim_product_count,
        (SELECT COUNT(*) FROM fact_sales) AS fact_sales_count
""", conn)
verify


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16884\3919101552.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  verify = pd.read_sql("""


,dim_date_count,dim_customer_count,dim_product_count,fact_sales_count
0,739,5878,4630,779407


In [20]:
# Sanity check: total revenue should match between Pandas and Postgres
pandas_total = df_clean['LineTotal'].sum()
pg_total = pd.read_sql("SELECT SUM(line_total) AS total FROM fact_sales", conn)['total'][0]

print(f"Total revenue (Pandas):   {pandas_total:,.2f}")
print(f"Total revenue (Postgres): {pg_total:,.2f}")
print(f"Match: {abs(pandas_total - float(pg_total)) < 1}")


Total revenue (Pandas):   17,374,804.25
Total revenue (Postgres): 17,374,804.25
Match: True


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16884\2654423669.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pg_total = pd.read_sql("SELECT SUM(line_total) AS total FROM fact_sales", conn)['total'][0]


In [21]:
cur.close()
conn.close()
print("Connection closed. Database is ready for Power BI.")


Connection closed. Database is ready for Power BI.
